# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs and survey metadata for predictors of household adoption of indigenous and modern knowledge in rangeland management in northern Kenya.

### Dataset Source
Dataset source (Croissant schema URL):

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Let's load metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)

# Display human-readable metadata
md = dataset.metadata
print(f"Dataset Title: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Spatial Coverage: {getattr(md, 'spatialCoverage', None)}")
print(f"Temporal Coverage: {getattr(md, 'temporalCoverage', None)}")
print(f"License: {getattr(md, 'license', None)}")

## 2. Data Overview

We'll inspect available record sets, their `@id`s, and included fields. All references use `@id` to remain schema-compliant and ensure reproducibility.

Let's list all record sets and their fields.

In [ ]:
# Enumerate all record sets defined in the dataset metadata
print("Record Sets (@id):")
record_sets = []
for rs in getattr(md, 'recordSet', []):
    print(f"  - {rs['@id']}")
    record_sets.append(rs['@id'])

# Print fields/columns within each record set, referenced by @id
print("\nFields per Record Set:")
for rs_id in record_sets:
    # Find the record set schema object
    record_set_obj = dataset.metadata.find_by_id(rs_id)
    if hasattr(record_set_obj, 'field'):
        field_ids = [f['@id'] for f in getattr(record_set_obj, 'field', [])]
        print(f"  {rs_id}:")
        for fid in field_ids:
            print(f"    - {fid}")
    else:
        print(f"  {rs_id}: [No fields declared]")

## 3. Data Extraction

We'll extract data from all available record sets into pandas DataFrames using the record set `@id`. For demonstration, we'll preview one DataFrame's columns and first rows.

If there are multiple record sets, we'll process each by their `@id`.

In [ ]:
# Collect records for each record set into DataFrames using their @id
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print column names of first available DataFrame (if any record sets exist)
if len(record_sets) > 0:
    example_rs_id = record_sets[0]
    print(f"Columns in record set '{example_rs_id}':")
    print(dataframes[example_rs_id].columns.tolist())
    print("\nFirst five rows:")
    display(dataframes[example_rs_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate typical EDA steps:
- Filtering rows by value in a numeric field (referenced by `@id` as per the schema)
- Normalizing that field
- Optionally grouping by a categorical or grouping field, also referenced by `@id`.

Please adjust the field `@id`s below to those that exist in the dataset's record sets.

In [ ]:
# Example: EDA on the first record set (if available and non-empty)
if len(record_sets) > 0 and not dataframes[example_rs_id].empty:
    df = dataframes[example_rs_id]

    # Automatically pick a numeric field using pandas dtype inference
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Analyzing numeric field (by @id): {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records where {numeric_field} > {threshold:.2f}")

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (e.g., those with object or category dtype and not unique for every row)
        group_field_candidates = [col for col in df.columns if (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])) and df[col].nunique() < len(df) // 2]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"\nGrouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization

Let's visualize the distribution of a selected numeric field or a group comparison (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if EDA produced data
if len(record_sets) > 0 and not dataframes[example_rs_id].empty:
    df = dataframes[example_rs_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field], bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field} (@id)")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook has demonstrated how to load FAIR^2 dataset metadata and records using the `mlcroissant` library, referencing all data entities using their `@id`.
- We explored available record sets and their fields, extracted data to pandas DataFrames, and performed typical data exploratory and visualization steps.
- You are encouraged to continue the analysis by mapping field `@id`s to their meanings using the Croissant schema or accompanying data dictionary for this dataset. This ensures all processing and interpretation remain schema-aware and transparent.

For more details on `mlcroissant`, visit the [official documentation](https://github.com/mlcommons/croissant).